# 08a — Dense Embedding Benchmark (Hue Foods RAG)

Notebook này đo lường và đánh giá các cấu hình dense vector space trên bộ dữ liệu ẩm thực Huế (572 chunks) và Golden Dataset V3 (45 câu hỏi chuẩn du lịch).

Mục tiêu là tạo bằng chứng kỹ thuật thực nghiệm để phục vụ lựa chọn mô hình, không thực hiện cutover production hay thay đổi cấu hình active.

## 1. Mục đích và Phạm vi (Relationship to Current Retrieval Profiles)

Hệ thống hiện có 3 profile truy xuất:
- `dense_only`: chỉ dùng vector dense cosine similarity.
- `hybrid_no_rerank`: dense top-30 kết hợp BM25 rescoring (được nghiên cứu tại Notebook 08b).
- `hybrid_rerank`: kết hợp dense + BM25 + CrossEncoder reranking (được nghiên cứu tại Notebook 08c & 08d).

Trong Notebook 08a, mọi cấu hình được cố định ở chế độ **`dense_only`** nhằm cô lập hoàn toàn biến số embedding model.

### Thiết lập đường dẫn backend

In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Không tìm thấy thư mục backend/. Hãy mở notebook từ repo root hoặc thư mục notebooks/."
    )
print(f"backend on path: {sys.path[0]}")

### Import các hàm backend benchmark

In [ ]:
from embedding.dense_benchmark import (
    E5_SMALL_SETTING,
    AUTHORIZED_DENSE_CANDIDATE_SETTINGS,
    ALL_DENSE_SETTINGS,
)
from evaluation.embedding_benchmark import (
    load_embedding_benchmark_inputs,
    snapshot_active_collection,
    run_embedding_benchmark,
    run_embedding_benchmarks,
    describe_embedding_benchmark_environment,
    settings_table,
    display_canonical_inputs,
    quality_table,
    category_table,
    comparison_table,
    latency_table,
    resource_table,
    failure_table,
    EMBEDDING_RESULTS_PATH,
)

## 2. Môi trường thực thi (Execution Profile)

Benchmark được chạy với cấu hình chuẩn:
- Thiết bị: **CPU FP32** (không quantization, không mixed precision).
- Batch size: **8 documents** khi indexing, **1 query** khi truy vấn.
- Lặp lại: 1 warm-up bị loại bỏ, sau đó **3 full repetitions** của 45 câu hỏi.
- Nguyên tắc: Không tự động retry, không shrink batch size, không đổi model fallback khi lỗi.

In [ ]:
env_info = describe_embedding_benchmark_environment()
env_info

## 3. Dữ liệu đầu vào chuẩn (Canonical Inputs & Relevance Definition)

Dữ liệu gồm 45 câu hỏi du lịch Golden V3 và 572 chunks chuẩn từ 91 file curated Markdown.
Độ liên quan (relevance) được định nghĩa là **binary exact match** theo cặp `(source, section)` đã khai báo trong evidence. Mỗi cặp evidence chỉ được ghi nhận điểm tối đa 1 lần trong Top 5.

In [ ]:
benchmark_inputs = load_embedding_benchmark_inputs()
display_canonical_inputs(benchmark_inputs)

## 4. Danh mục 8 Cấu hình và Trạng thái Thực thi (Settings & Isolation)

Catalog gồm 8 cấu hình dense embedding trong các không gian vector cô lập riêng biệt. Trong đó 5 cấu hình được cấp quyền chạy thực nghiệm (Authorized), 3 cấu hình còn lại ở trạng thái trì hoãn (Deferred). Active collection `hue_foods_e5_small_384` được giữ read-only tuyệt đối.

In [ ]:
settings_df = settings_table()
settings_df

### Chụp snapshot active collection trước khi chạy

In [ ]:
active_before = snapshot_active_collection(benchmark_inputs)
active_before

## 5. Chạy mô hình Control (E5-small 384D)

Chạy E5-small làm mốc đối chứng (control baseline) trên collection cô lập `hue_foods_08a_e5_small_384`.

In [ ]:
control_result = run_embedding_benchmark(
    E5_SMALL_SETTING,
    benchmark_inputs,
    expected_active_snapshot=active_before,
)
control_result.summary

### Kiểm tra active collection sau control run

In [ ]:
active_after_control = snapshot_active_collection(benchmark_inputs)
assert active_after_control == active_before, "Active collection bị thay đổi sau control run!"
print("Active collection an toàn, không bị ảnh hưởng.")

## 6. Chạy tuần tự 4 mô hình Candidates được ủy quyền (Authorized Candidates)

Chạy tuần tự 4 candidate models được ủy quyền (`MiniLM-L12 384D`, `Huydang DEk21 768D`, `E5-base 768D`, `Qwen3 384D`). Mỗi candidate sau khi hoàn tất sẽ giải phóng model và bộ nhớ trước khi nạp mô hình kế tiếp.

In [ ]:
candidate_results = []

for result in run_embedding_benchmarks(
    AUTHORIZED_DENSE_CANDIDATE_SETTINGS,
    benchmark_inputs,
    control_result=control_result,
    expected_active_snapshot=active_before,
):
    candidate_results.append(result)
    print(f"✓ Đã chạy xong: {result.setting.setting_label} -> Trạng thái: {result.status}")

## 7. Tổng hợp và So sánh Kết quả (Comparison and Conclusion)

### Bảng chất lượng truy xuất tổng thể (Recall@5, MRR@5, nDCG@5)

In [ ]:
quality_df = quality_table(control_result, candidate_results)
quality_df

### Bảng chi tiết theo 9 Categories (Hits, Recall@5, MRR@5, nDCG@5, Delta nDCG, Guardrail)

In [ ]:
category_df = category_table(control_result, candidate_results)
category_df

### Bảng so sánh Bootstrap, Guardrails và Quyết định Clear Gain

In [ ]:
comp_df = comparison_table(control_result, candidate_results)
comp_df

### Bảng độ trễ (Latency: Cold Load, Document Embedding, Query p50/p95, Retrieval p50/p95)

In [ ]:
latency_df = latency_table(control_result, candidate_results)
latency_df

### Bảng tài nguyên bộ nhớ (Memory RSS) và Truncation

In [ ]:
resource_df = resource_table(control_result, candidate_results)
resource_df

### Bảng trạng thái thực thi và ghi nhận lỗi (nếu có)

In [ ]:
failure_df = failure_table(control_result, candidate_results)
failure_df

### Xác nhận file CSV bền vững và an toàn Active Collection

In [ ]:
active_final = snapshot_active_collection(benchmark_inputs)
assert active_final == active_before, "Active collection bị thay đổi sau benchmark!"

print(f"✓ Dữ liệu benchmark đã lưu tại: {EMBEDDING_RESULTS_PATH}")
print(f"✓ Active collection {active_final['collection_name']} giữ nguyên {active_final['points_count']} points (read-only tuyệt đối).")

### Kết luận kỹ thuật và Đánh giá Trade-offs

- **Lighter vs Heavier trade-off**:
  - `E5-small 384D (control)` là mô hình đối chứng nhẹ nhất với hiệu năng nDCG@5 và MRR@5 ổn định, độ trễ truy vấn thấp (~24 ms) và mức tiêu thụ RAM nhỏ (~1.57 GB).
  - `MiniLM-L12 384D` bị giới hạn max_length=128 gây cắt ngắn 83 đoạn văn bản, làm giảm chất lượng rõ rệt.
  - `Huydang DEk21 768D` và `E5-base 768D` cho góc nhìn so sánh giữa biểu diễn chuyên biệt tiếng Việt (PhoBERT) và mô hình đa ngôn ngữ lớn hơn.
  - `Qwen3 384D` thể hiện khả năng biểu diễn của kiến trúc 0.6B LLM-backbone khi nén xuống 384D vector space.

> **Lưu ý**: Kết quả thực nghiệm trên là bằng chứng kỹ thuật phục vụ so sánh và lựa chọn model trong Phase 8. Notebook này **không** tự động cutover production hay thay đổi cấu hình active của hệ thống.